# 07 — Traditional ML & Ensemble (No Deep Learning)

Baseline: flatten raw pixel values from resized images, train RF/DT/SVM/LR classifiers. No CNN feature extractor — this is the pure classical ML comparison group.

## Section 1: Imports & Configuration

In [ ]:
import os
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import label_binarize
import joblib
from tqdm import tqdm
print("✓ Imports complete")

## Section 2: Constants & Hyperparameters

In [ ]:
NOTEBOOK_NAME    = "07_TraditionalML_Ensemble"

DATASET_PATH     = "../MRI_DATASET/"
TRAIN_DIR        = DATASET_PATH + "Training/"
TEST_DIR         = DATASET_PATH + "Testing/"

CLASS_NAMES      = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES      = 4

# Resize to 64×64 for manageable feature vector (12288 features)
IMG_HEIGHT       = 64
IMG_WIDTH        = 64
CHANNELS         = 3

RANDOM_SEED      = 42
SAVED_MODELS_DIR = "../saved_models/"
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("✓ Constants configured")
print(f"  Image size for features : {IMG_HEIGHT}×{IMG_WIDTH}×{CHANNELS}")
print(f"  Feature vector size     : {IMG_HEIGHT * IMG_WIDTH * CHANNELS}")
print(f"  Classes                 : {CLASS_NAMES}")

## Section 3: Data Loading & Verification

In [ ]:
def load_images_from_dir(base_dir, class_names, img_h, img_w):
    images, labels = [], []
    for idx, cls in enumerate(class_names):
        cls_path = os.path.join(base_dir, cls)
        if not os.path.exists(cls_path):
            print(f"  WARNING: {cls_path} not found")
            continue
        files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"  {cls}: {len(files)} images")
        for fname in tqdm(files, desc=cls, leave=False):
            img = cv2.imread(os.path.join(cls_path, fname))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_w, img_h))
            images.append(img)
            labels.append(idx)
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32)

print("Loading training images...")
X_train_raw, y_train = load_images_from_dir(TRAIN_DIR, CLASS_NAMES, IMG_HEIGHT, IMG_WIDTH)
print(f"✓ Train: {X_train_raw.shape}")

print("\nLoading test images...")
X_test_raw, y_test = load_images_from_dir(TEST_DIR, CLASS_NAMES, IMG_HEIGHT, IMG_WIDTH)
print(f"✓ Test : {X_test_raw.shape}")

print("\n" + "=" * 50)
print("DATA VERIFICATION")
print(f"Class order   : {CLASS_NAMES}")
print(f"Train samples : {X_train_raw.shape[0]}")
print(f"Test samples  : {X_test_raw.shape[0]}")
for u, c in zip(*np.unique(y_train, return_counts=True)):
    print(f"  {CLASS_NAMES[u]}: {c} training images")
print("=" * 50)

## Section 4: Data Preprocessing & Augmentation

In [ ]:
# Normalise to [0, 1] then flatten to 1D feature vectors
X_train = (X_train_raw / 255.0).reshape(X_train_raw.shape[0], -1)
X_test  = (X_test_raw  / 255.0).reshape(X_test_raw.shape[0],  -1)
n_features = X_train.shape[1]

print(f"✓ Normalised to [0, 1] and flattened")
print(f"  Train shape : {X_train.shape}")
print(f"  Test shape  : {X_test.shape}")
print(f"  Features    : {n_features}")

## Section 5: Model Definition

In [ ]:
rf  = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
dt  = DecisionTreeClassifier(random_state=RANDOM_SEED)
svm = SVC(probability=True, random_state=RANDOM_SEED, kernel='rbf', C=1.0)
lr  = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, C=0.1, n_jobs=-1)
ensemble_clf = VotingClassifier(
    estimators=[('rf', rf), ('dt', dt), ('svm', svm)],
    voting='soft',
    n_jobs=-1
)
print("✓ Classifiers: RF, DT, SVM, Logistic Regression, Soft-Vote Ensemble")

## Section 6: Model Training

In [ ]:
print("Training Random Forest...")
rf.fit(X_train, y_train)
print("✓ Random Forest trained")

print("Training Decision Tree...")
dt.fit(X_train, y_train)
print("✓ Decision Tree trained")

print("Training SVM (may take several minutes)...")
svm.fit(X_train, y_train)
print("✓ SVM trained")

print("Training Logistic Regression...")
lr.fit(X_train, y_train)
print("✓ Logistic Regression trained")

print("Training Soft-Vote Ensemble...")
ensemble_clf.fit(X_train, y_train)
print("✓ Ensemble trained")

## Section 7: Model Evaluation

In [ ]:
def evaluate_sklearn_model(clf, X_test, y_test, model_name="Model"):
    """Standard evaluation for sklearn classifiers."""    y_pred       = clf.predict(X_test)
    y_pred_proba = clf.predict_proba(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm  = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_name} — Confusion Matrix (Acc: {acc:.4f})')
    plt.ylabel('Actual'); plt.xlabel('Predicted')
    plt.tight_layout(); plt.show()
    print(f"\n{model_name} — Classification Report")
    print("=" * 60)
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))
    y_true_bin = label_binarize(y_test, classes=[0, 1, 2, 3])
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        plt.plot(fpr, tpr, label=f'{cls} (AUC = {auc(fpr, tpr):.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} — ROC Curve'); plt.legend(loc='lower right')
    plt.tight_layout(); plt.show()
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        p, r, _ = precision_recall_curve(y_true_bin[:, i], y_pred_proba[:, i])
        ap = average_precision_score(y_true_bin[:, i], y_pred_proba[:, i])
        plt.plot(r, p, label=f'{cls} (AP = {ap:.2f})')
    plt.xlabel('Recall'); plt.ylabel('Precision')
    plt.title(f'{model_name} — Precision-Recall Curve'); plt.legend(loc='upper right')
    plt.tight_layout(); plt.show()
    return acc, y_pred, y_pred_proba

rf_acc,  _, _ = evaluate_sklearn_model(rf,  X_test, y_test, "Random Forest (raw pixels)")
dt_acc,  _, _ = evaluate_sklearn_model(dt,  X_test, y_test, "Decision Tree (raw pixels)")
svm_acc, _, _ = evaluate_sklearn_model(svm, X_test, y_test, "SVM (raw pixels)")
lr_acc,  _, _ = evaluate_sklearn_model(lr,  X_test, y_test, "Logistic Regression (raw pixels)")
ens_acc, _, _ = evaluate_sklearn_model(ensemble_clf, X_test, y_test, "Soft-Vote Ensemble (raw pixels)")

print("\n" + "=" * 50)
print("ACCURACY COMPARISON — Traditional ML on raw pixels")
print(f"  Random Forest       : {rf_acc:.4f}")
print(f"  Decision Tree       : {dt_acc:.4f}")
print(f"  SVM                 : {svm_acc:.4f}")
print(f"  Logistic Regression : {lr_acc:.4f}")
print(f"  Soft-Vote Ensemble  : {ens_acc:.4f}")
print("=" * 50)

## Section 8: Save Model

In [ ]:
joblib.dump(rf,  SAVED_MODELS_DIR + 'traditional_rf.pkl')
joblib.dump(dt,  SAVED_MODELS_DIR + 'traditional_dt.pkl')
joblib.dump(svm, SAVED_MODELS_DIR + 'traditional_svm.pkl')
joblib.dump(lr,  SAVED_MODELS_DIR + 'traditional_lr.pkl')
joblib.dump(ensemble_clf, SAVED_MODELS_DIR + 'traditional_ensemble_model.pkl')
print(f"✓ All models saved to {SAVED_MODELS_DIR}")

## Section 9: Results Summary

In [ ]:
print("=" * 60)
print(f"NOTEBOOK: {NOTEBOOK_NAME}")
print(f"Dataset  : {X_train.shape[0]} training + {X_test.shape[0]} test images")
print(f"Classes  : {CLASS_NAMES}")
print(f"Image size: {IMG_HEIGHT}×{IMG_WIDTH}×{CHANNELS} → {n_features} features (flattened)")
print(f"Seed     : {RANDOM_SEED}")
print("-" * 60)
print(f"  Random Forest       : {rf_acc:.4f}")
print(f"  Decision Tree       : {dt_acc:.4f}")
print(f"  SVM                 : {svm_acc:.4f}")
print(f"  Logistic Regression : {lr_acc:.4f}")
print(f"  Soft-Vote Ensemble  : {ens_acc:.4f}")
print("=" * 60)
print("Saved models location:", SAVED_MODELS_DIR)